<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet%2011%20PBL/Praktikum%205%20(JS11)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 0 - Import Library & Setup Dataset

In [2]:
# Import library yang dibutuhkan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Memuat dataset cafes.csv dan reviews.csv
cafes_df = pd.read_csv('cafes.csv')
reviews_df = pd.read_csv('reviews.csv')

print("Jumlah data cafe:", cafes_df.shape)
print("Jumlah data review:", reviews_df.shape)

Jumlah data cafe: (99, 12)
Jumlah data review: (2230, 8)


#Langkah 1 - Load Data Teks & Inspeksi Data Review

In [3]:
# Membersihkan data dari missing value pada teks dan rating
reviews_clean = reviews_df.dropna(subset=['text', 'rating']).copy()

# Membuat list tuple (teks_review, label_kategori)
# Label: 1 jika rating >= 4.5 (Puas/Positif), 0 jika < 4.5 (Standar/Negatif)
reviews_clean['label'] = (reviews_clean['rating'] >= 4.5).astype(int)

# Membagi dataset menjadi training dan testing secara manual (80:20)
train_size = int(0.8 * len(reviews_clean))
train_reviews = reviews_clean.iloc[:train_size].values
test_reviews = reviews_clean.iloc[train_size:].values

print(f"Jumlah Data Training: {len(train_reviews)}")
print(f"Jumlah Data Testing: {len(test_reviews)}")

# Inspeksi data teks secara acak (analog dengan random_img_viz)
def random_text_viz(data_list):
    rand_num = np.random.randint(0, len(data_list))
    text_content = data_list[rand_num][4] # kolom text
    label_val = data_list[rand_num][-1]   # kolom label
    print(f"Indeks Data : {rand_num}")
    print(f"Label       : {label_val} ('Puas' jika 1, 'Standar' jika 0)")
    print(f"Teks Review : \"{text_content}\"")

random_text_viz(train_reviews)

Jumlah Data Training: 1784
Jumlah Data Testing: 446
Indeks Data : 195
Label       : 1 ('Puas' jika 1, 'Standar' jika 0)
Teks Review : "Tempat nya luas enak, colokan banyak, makanan dan minuman affordable. Sejuk, not crowded."


#Langkah 3 - Pra-pengolahan Data Teks (Standardisasi & Encoding Label)

In [4]:
def preprocess_text(data_list):
    processed_list = []
    for item in data_list:
        text_content = str(item[4]) # Kolom text
        label = item[-1]            # Kolom label (0 atau 1)

        # Pra-pengolahan teks sederhana (lowercase & strip)
        std_text = text_content.lower().strip()

        processed_list.append((std_text, label))
    return processed_list

# Melakukan pra-pengolahan pada data training dan testing
train_std_list = preprocess_text(train_reviews)
test_std_list = preprocess_text(test_reviews)

print("Contoh hasil pra-pengolahan teks indeks ke-0:")
print(train_std_list[0])

Contoh hasil pra-pengolahan teks indeks ke-0:
('hore!', 1)


#Langkah 4 - Ekstraksi Fitur Teks (Menggantikan Average Brightness)

In [5]:
# Fungsi untuk mengekstrak fitur numerik dari teks (jumlah kata)
def text_feature_extraction(text):
    words = text.split()
    word_count = len(words)
    return word_count

# Pengecekan ekstraksi fitur pada teks secara acak
rand_idx = np.random.randint(0, len(train_std_list))
sample_text = train_std_list[rand_idx][0]
feature_val = text_feature_extraction(sample_text)

print(f"Teks: \"{sample_text}\"")
print(f"Fitur (Jumlah Kata): {feature_val}")

Teks: "pelayanan bagus dan ramah apalagi barista yg pake kacamata tatoan kecil2, latte artnya bagus, kopinya enak juga, worth to try dan cocok buat nugas lamaan"
Fitur (Jumlah Kata): 25


#Langkah 5 - Klasifikasi Menggunakan Threshold Sederhana

In [6]:
def predict_text_label(text, threshold):
    feature_val = text_feature_extraction(text)
    pred = 0
    # Jika jumlah kata melebihi threshold, predikisi kelas 1
    if feature_val > threshold:
        pred = 1
    return pred

# Uji coba prediksi pada data training secara acak
rand_idx = np.random.randint(0, len(train_std_list))
sample_item = train_std_list[rand_idx]
pred_label = predict_text_label(sample_item[0], threshold=15)

print(f"Teks Review : \"{sample_item[0]}\"")
print(f"Aktual Label: {sample_item[1]}")
print(f"Prediksi    : {pred_label}")

Teks Review : "dijamin gk nyesel selalu ruame dan enak"
Aktual Label: 1
Prediksi    : 0


#Langkah 6 - Evaluasi Manual & Pengujian pada Data Testing

In [7]:
# Fungsi evaluasi akurasi manual
def evaluate_threshold(data_list, threshold):
    miss_count = 0
    for item in data_list:
        text = item[0]
        actual_label = item[1]
        pred_label = predict_text_label(text, threshold)
        if pred_label != actual_label:
            miss_count += 1
    total = len(data_list)
    accuracy = (total - miss_count) / total
    print(f"Akurasi Threshold (thres={threshold}): {accuracy:.4f}")

# Evaluasi pada data training dan testing
print("Evaluasi Data Training:")
evaluate_threshold(train_std_list, threshold=10)

print("\nEvaluasi Data Testing:")
evaluate_threshold(test_std_list, threshold=10)

Evaluasi Data Training:
Akurasi Threshold (thres=10): 0.4355

Evaluasi Data Testing:
Akurasi Threshold (thres=10): 0.4417


#Langkah 4 Alternatif & 5 - Ekstraksi Fitur Tabular & Model SVM

In [8]:
# Fungsi untuk menyusun fitur ekstraksi teks ke dalam DataFrame tabular
def extract_tabular_features(data_list):
    features = []
    labels = []
    for item in data_list:
        text = item[0]
        label = item[1]
        word_count = text_feature_extraction(text)
        features.append(word_count)
        labels.append(label)

    data = np.column_stack((features, labels))
    df = pd.DataFrame(data, columns=['WORD_COUNT', 'LABELS'])
    return df

# Ekstraksi untuk train dan test DataFrame
train_avg_img = extract_tabular_features(train_std_list)
test_avg_img = extract_tabular_features(test_std_list)

print("Shape Train Tabular:", train_avg_img.shape)
display(train_avg_img.head())

# Mempersiapkan data untuk Scikit-Learn SVM
X_train = train_avg_img.iloc[:, 0].values.reshape(-1, 1)
y_train = train_avg_img.iloc[:, 1].values

X_test = test_avg_img.iloc[:, 0].values.reshape(-1, 1)
y_test = test_avg_img.iloc[:, 1].values

# Inisialisasi dan Training Model SVM
svm_model = SVC(kernel='rbf')
svm_model.fit(X_train, y_train)

print("Model SVM Berhasil Dilatih menggunakan Fitur Teks Review Kafe!")

Shape Train Tabular: (1784, 2)


,WORD_COUNT,LABELS
0,1,1
1,25,1
2,20,1
3,7,1
4,12,1


Model SVM Berhasil Dilatih menggunakan Fitur Teks Review Kafe!


#Langkah 6 - Evaluasi Model SVM

In [9]:
# Prediksi dan evaluasi menggunakan model SVM
y_train_pred = svm_model.predict(X_train)
y_test_pred = svm_model.predict(X_test)

acc_train = accuracy_score(y_train, y_train_pred)
acc_test = accuracy_score(y_test, y_test_pred)

print(f"Akurasi SVM pada Train Data: {acc_train:.4f}")
print(f"Akurasi SVM pada Test Data : {acc_test:.4f}")

Akurasi SVM pada Train Data: 0.8072
Akurasi SVM pada Test Data : 0.8274
